<a href="https://colab.research.google.com/github/Farzanehnaderi/Tehran_ENUI_Urban_Monitoring_GEE/blob/main/Tehran_Temporal_Persistence_2015_2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# Tehran Temporal Persistence Analysis
# VIIRS Monthly Nighttime Lights
# Period: 2015–2025
# ==========================================

import ee
import geemap
import numpy as np
import pandas as pd

ee.Authenticate()
ee.Initialize(project="bold-physics-478216-b4")

print("Google Earth Engine initialized successfully.")

In [ ]:
# ==========================================
# Tehran Province ROI
# ==========================================

tehran = ee.FeatureCollection(
    "projects/bold-physics-478216-b4/assets/Teharn_province"
)

roi = tehran.geometry()

print("Number of features:", tehran.size().getInfo())
print("ROI loaded successfully.")

In [ ]:
# ==========================================
# Display ROI
# ==========================================

Map = geemap.Map(basemap="SATELLITE")

Map.centerObject(roi, 8)

Map.addLayer(
    tehran.style(
        color="red",
        fillColor="00000000",
        width=2
    ),
    {},
    "Tehran Province"
)

Map.add_layer_control()

Map

In [ ]:
# ==========================================
# VIIRS Monthly Nighttime Lights
# Period: 2015–2025
# ==========================================

viirs_monthly = (
    ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMCFG")
    .filterBounds(roi)
    .filterDate("2015-01-01", "2026-01-01")
)

print(
    "Number of VIIRS monthly images:",
    viirs_monthly.size().getInfo()
)

In [ ]:
# ==========================================
# Cell 4 — Monthly VIIRS Time Series
# ==========================================

def extract_monthly_stats(image):
    date = ee.Date(image.get("system:time_start"))

    stats = image.select(["avg_rad", "cf_cvg"]).reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=500,
        maxPixels=1e9
    )

    return ee.Feature(None, {
        "date": date.format("YYYY-MM-dd"),
        "year": date.get("year"),
        "month": date.get("month"),
        "avg_rad_mean": stats.get("avg_rad"),
        "cf_cvg_mean": stats.get("cf_cvg")
    })


monthly_stats = ee.FeatureCollection(
    viirs_monthly.map(extract_monthly_stats)
)

monthly_data = monthly_stats.getInfo()["features"]

monthly_df = pd.DataFrame([
    feature["properties"] for feature in monthly_data
])

monthly_df = monthly_df.sort_values(
    ["year", "month"]
).reset_index(drop=True)

print("Number of monthly records:", len(monthly_df))
monthly_df.head(12)

In [ ]:
# ==========================================
# Cell 5 — Monthly Image Count by Year
# ==========================================

yearly_counts = []

for year in range(2015, 2026):
    start_date = f"{year}-01-01"
    end_date = f"{year + 1}-01-01"

    count = (
        viirs_monthly
        .filterDate(start_date, end_date)
        .size()
        .getInfo()
    )

    yearly_counts.append({
        "year": year,
        "image_count": count
    })

yearly_counts_df = pd.DataFrame(yearly_counts)

yearly_counts_df

In [ ]:
# ==========================================
# Cell 6 — VIIRS Data Quality Check
# ==========================================

quality_df = monthly_df[
    ["date", "year", "month", "avg_rad_mean", "cf_cvg_mean"]
].copy()

# ماه‌هایی که هیچ پوشش cloud-free ثبت نشده
missing_months = quality_df[
    quality_df["cf_cvg_mean"] <= 0
]

print("Total months:", len(quality_df))
print("Months with cf_cvg = 0:", len(missing_months))

print("\nMonths with no valid coverage:")
print(
    missing_months[
        ["date", "avg_rad_mean", "cf_cvg_mean"]
    ].to_string(index=False)
)

In [ ]:
# ==========================================
# Cell 7 — Detailed Quality Check
# ==========================================

def check_viirs_month(date_start, date_end):

    image = (
        viirs_monthly
        .filterDate(date_start, date_end)
        .first()
    )

    stats = image.select(["avg_rad", "cf_cvg"]).reduceRegion(
        reducer=ee.Reducer.minMax()
            .combine(
                reducer2=ee.Reducer.mean(),
                sharedInputs=True
            ),
        geometry=roi,
        scale=500,
        maxPixels=1e9
    )

    return stats.getInfo()


# یک ماه سالم
print("May 2019:")
print(check_viirs_month("2019-05-01", "2019-06-01"))

# یک ماه مشکل‌دار
print("\nJune 2019:")
print(check_viirs_month("2019-06-01", "2019-07-01"))

In [ ]:
# ==========================================
# Cell 7 — Prepare VIIRS for Temporal Interpolation
# ==========================================

def prepare_viirs_image(image):

    avg_rad = image.select("avg_rad")
    cf_cvg = image.select("cf_cvg")

    # فقط پیکسل‌هایی که حداقل یک مشاهده cloud-free دارند معتبرند
    valid_mask = cf_cvg.gt(0)

    # داده‌های نامعتبر به صورت masked باقی می‌مانند
    avg_rad_valid = avg_rad.updateMask(valid_mask)

    return avg_rad_valid.rename("avg_rad").copyProperties(
        image,
        ["system:time_start"]
    )


viirs_valid = viirs_monthly.map(prepare_viirs_image)

print(
    "Number of prepared VIIRS images:",
    viirs_valid.size().getInfo()
)

In [ ]:
# ==========================================
# Cell 8 — Temporal Linear Interpolation
# Using cf_cvg as validity mask
# ==========================================

def interpolate_image(image):

    image = ee.Image(image)

    # --------------------------------------
    # Current month's radiance
    # --------------------------------------

    current_rad = image.select("avg_rad")

    # --------------------------------------
    # Validity mask
    # cf_cvg > 0 means valid observation
    # --------------------------------------

    current_mask = image.select("cf_cvg").gt(0)

    current_rad = current_rad.updateMask(current_mask)

    current_time = ee.Number(
        image.get("system:time_start")
    )

    # --------------------------------------
    # Previous image
    # --------------------------------------

    before = (
        viirs_monthly
        .filter(
            ee.Filter.lt(
                "system:time_start",
                current_time
            )
        )
        .sort("system:time_start", False)
        .first()
    )

    # --------------------------------------
    # Next image
    # --------------------------------------

    after = (
        viirs_monthly
        .filter(
            ee.Filter.gt(
                "system:time_start",
                current_time
            )
        )
        .sort("system:time_start", True)
        .first()
    )

    before = ee.Image(before)
    after = ee.Image(after)

    # --------------------------------------
    # Radiance of previous and next month
    # --------------------------------------

    before_rad = before.select("avg_rad")
    after_rad = after.select("avg_rad")

    # --------------------------------------
    # Validity masks for before/after
    # --------------------------------------

    before_rad = before_rad.updateMask(
        before.select("cf_cvg").gt(0)
    )

    after_rad = after_rad.updateMask(
        after.select("cf_cvg").gt(0)
    )

    # --------------------------------------
    # Time of previous and next image
    # --------------------------------------

    before_time = ee.Number(
        before.get("system:time_start")
    )

    after_time = ee.Number(
        after.get("system:time_start")
    )

    # --------------------------------------
    # Temporal interpolation fraction
    # --------------------------------------

    fraction = (
        current_time
        .subtract(before_time)
        .divide(
            after_time.subtract(before_time)
        )
    )

    # --------------------------------------
    # Linear interpolation
    # --------------------------------------

    interpolated = (
        before_rad
        .add(
            after_rad
            .subtract(before_rad)
            .multiply(fraction)
        )
        .rename("avg_rad")
    )

    # --------------------------------------
    # Fill missing pixels
    # --------------------------------------

    filled = (
        current_rad
        .unmask(interpolated)
        .rename("avg_rad")
    )

    return filled.copyProperties(
        image,
        ["system:time_start"]
    )


viirs_interpolated = ee.ImageCollection(
    viirs_monthly.map(interpolate_image)
)

print(
    "Number of interpolated images:",
    viirs_interpolated.size().getInfo()
)

In [ ]:
# ==========================================
# Cell 9 — Validate Interpolation
# May, June and July 2019
# ==========================================

def get_month_mean(date_start, date_end, collection):

    image = (
        collection
        .filterDate(date_start, date_end)
        .first()
    )

    stats = image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=500,
        maxPixels=1e9
    )

    return stats.getInfo()


may_original = get_month_mean(
    "2019-05-01",
    "2019-06-01",
    viirs_monthly
)

june_original = get_month_mean(
    "2019-06-01",
    "2019-07-01",
    viirs_monthly
)

july_original = get_month_mean(
    "2019-07-01",
    "2019-08-01",
    viirs_monthly
)

june_interpolated = get_month_mean(
    "2019-06-01",
    "2019-07-01",
    viirs_interpolated
)

print("May 2019 - Original:")
print(may_original)

print("\nJune 2019 - Original:")
print(june_original)

print("\nJuly 2019 - Original:")
print(july_original)

print("\nJune 2019 - After interpolation:")
print(june_interpolated)

In [ ]:
# ==========================================
# Cell 10 — Validate Another Missing Month
# July 2022
# ==========================================

june_2022 = get_month_mean(
    "2022-06-01",
    "2022-07-01",
    viirs_monthly
)

july_2022 = get_month_mean(
    "2022-07-01",
    "2022-08-01",
    viirs_monthly
)

august_2022 = get_month_mean(
    "2022-08-01",
    "2022-09-01",
    viirs_monthly
)

july_2022_interpolated = get_month_mean(
    "2022-07-01",
    "2022-08-01",
    viirs_interpolated
)

print("June 2022 - Original:")
print(june_2022)

print("\nJuly 2022 - Original:")
print(july_2022)

print("\nAugust 2022 - Original:")
print(august_2022)

print("\nJuly 2022 - After interpolation:")
print(july_2022_interpolated)

In [ ]:
# ==========================================
# Cell 11 — Detect Temporal Gaps
# 2015–2025
# ==========================================

# فقط ماه‌هایی که پوشش معتبر ندارند
missing_months = monthly_df[
    monthly_df["cf_cvg_mean"] <= 0
].copy()

missing_months = missing_months[
    ["date", "year", "month", "avg_rad_mean", "cf_cvg_mean"]
].reset_index(drop=True)

print("Total missing months:", len(missing_months))

print("\nMissing months:")
print(
    missing_months.to_string(index=False)
)


# ==========================================
# بررسی فاصله زمانی بین ماه‌های Missing
# ==========================================

missing_dates = pd.to_datetime(
    missing_months["date"]
)

print("\n------------------------------------------")
print("Gap analysis")
print("------------------------------------------")

for i in range(len(missing_dates)):

    current_date = missing_dates.iloc[i]

    if i == 0:
        print(
            current_date.strftime("%Y-%m"),
            "→ first missing month"
        )
        continue

    previous_date = missing_dates.iloc[i - 1]

    month_difference = (
        (current_date.year - previous_date.year) * 12
        + (current_date.month - previous_date.month)
    )

    if month_difference == 1:
        print(
            current_date.strftime("%Y-%m"),
            "→ consecutive with previous missing month"
        )
    else:
        print(
            current_date.strftime("%Y-%m"),
            f"→ gap of {month_difference - 1} valid month(s)"
        )

In [ ]:
# ==========================================
# Cell 12 — Create Valid VIIRS Collection
# ==========================================

# Dates of valid months
valid_dates = (
    monthly_df[
        monthly_df["cf_cvg_mean"] > 0
    ]["date"]
    .astype(str)
    .tolist()
)

# Convert dates to Earth Engine timestamps
valid_millis = [
    int(pd.Timestamp(date).timestamp() * 1000)
    for date in valid_dates
]

# Keep only images corresponding to valid months
viirs_valid = viirs_monthly.filter(
    ee.Filter.inList(
        "system:time_start",
        valid_millis
    )
)

print(
    "Total VIIRS images:",
    viirs_monthly.size().getInfo()
)

print(
    "Valid VIIRS images:",
    viirs_valid.size().getInfo()
)

print(
    "Missing VIIRS images:",
    viirs_monthly.size().getInfo()
    - viirs_valid.size().getInfo()
)

In [ ]:
# ==========================================
# Cell 13 — Robust Temporal Interpolation
# ==========================================

def interpolate_image(image):

    image = ee.Image(image)

    # Current month
    current_rad = image.select("avg_rad")
    current_valid = image.select("cf_cvg").gt(0)

    # Mask invalid observations
    current_rad = current_rad.updateMask(current_valid)

    current_time = ee.Number(
        image.get("system:time_start")
    )

    # ------------------------------------------
    # Nearest valid image BEFORE current month
    # ------------------------------------------

    before = (
        viirs_valid
        .filter(
            ee.Filter.lt(
                "system:time_start",
                current_time
            )
        )
        .sort("system:time_start", False)
        .first()
    )

    # ------------------------------------------
    # Nearest valid image AFTER current month
    # ------------------------------------------

    after = (
        viirs_valid
        .filter(
            ee.Filter.gt(
                "system:time_start",
                current_time
            )
        )
        .sort("system:time_start", True)
        .first()
    )

    before = ee.Image(before)
    after = ee.Image(after)

    # ------------------------------------------
    # Radiance of neighboring valid months
    # ------------------------------------------

    before_rad = before.select("avg_rad")
    after_rad = after.select("avg_rad")

    # ------------------------------------------
    # Time of neighboring observations
    # ------------------------------------------

    before_time = ee.Number(
        before.get("system:time_start")
    )

    after_time = ee.Number(
        after.get("system:time_start")
    )

    # ------------------------------------------
    # Temporal interpolation factor
    # ------------------------------------------

    fraction = (
        current_time
        .subtract(before_time)
        .divide(
            after_time.subtract(before_time)
        )
    )

    # ------------------------------------------
    # Linear interpolation
    # ------------------------------------------

    interpolated = (
        before_rad
        .add(
            after_rad
            .subtract(before_rad)
            .multiply(fraction)
        )
        .rename("avg_rad")
    )

    # ------------------------------------------
    # Fill only invalid pixels/months
    # ------------------------------------------

    filled = (
        current_rad
        .unmask(interpolated)
        .rename("avg_rad")
    )

    return (
        filled
        .copyProperties(
            image,
            ["system:time_start"]
        )
    )


# Apply interpolation to all 132 months
viirs_interpolated = ee.ImageCollection(
    viirs_monthly.map(interpolate_image)
)

print(
    "Original images:",
    viirs_monthly.size().getInfo()
)

print(
    "Interpolated images:",
    viirs_interpolated.size().getInfo()
)

In [ ]:
# ==========================================
# Cell 14 — Validate Robust Interpolation
# ==========================================

# ------------------------------------------
# Function to calculate monthly mean
# ------------------------------------------

def get_month_mean_simple(date_start, date_end, collection):

    image = (
        collection
        .filterDate(date_start, date_end)
        .first()
    )

    stats = image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=500,
        maxPixels=1e9
    )

    return stats.getInfo()


# ------------------------------------------
# Original values
# ------------------------------------------

may_2022 = get_month_mean_simple(
    "2022-05-01",
    "2022-06-01",
    viirs_monthly
)

june_2022 = get_month_mean_simple(
    "2022-06-01",
    "2022-07-01",
    viirs_monthly
)

july_2022 = get_month_mean_simple(
    "2022-07-01",
    "2022-08-01",
    viirs_monthly
)

august_2022 = get_month_mean_simple(
    "2022-08-01",
    "2022-09-01",
    viirs_monthly
)


# ------------------------------------------
# Interpolated values
# ------------------------------------------

may_2022_int = get_month_mean_simple(
    "2022-05-01",
    "2022-06-01",
    viirs_interpolated
)

june_2022_int = get_month_mean_simple(
    "2022-06-01",
    "2022-07-01",
    viirs_interpolated
)

july_2022_int = get_month_mean_simple(
    "2022-07-01",
    "2022-08-01",
    viirs_interpolated
)

august_2022_int = get_month_mean_simple(
    "2022-08-01",
    "2022-09-01",
    viirs_interpolated
)


# ------------------------------------------
# Print results
# ------------------------------------------

print("========== ORIGINAL ==========")

print("May 2022   :", may_2022)
print("June 2022  :", june_2022)
print("July 2022  :", july_2022)
print("August 2022:", august_2022)


print("\n========== INTERPOLATED ==========")

print("May 2022   :", may_2022_int)
print("June 2022  :", june_2022_int)
print("July 2022  :", july_2022_int)
print("August 2022:", august_2022_int)

In [ ]:
# ==========================================
# Cell 15 — VIIRS Radiance Distribution
# Valid Months: 2015–2025
# ==========================================

# Calculate statistics for every valid monthly image
def get_image_stats(image):

    stats = image.select("avg_rad").reduceRegion(
        reducer=ee.Reducer.percentile(
            [1, 5, 10, 25, 50, 75, 90, 95, 99]
        ).combine(
            reducer2=ee.Reducer.mean(),
            sharedInputs=True
        ),
        geometry=roi,
        scale=500,
        maxPixels=1e9
    )

    date = ee.Date(
        image.get("system:time_start")
    )

    return ee.Feature(None, {
        "date": date.format("YYYY-MM-dd"),
        "mean": stats.get("avg_rad_mean"),
        "p01": stats.get("avg_rad_p1"),
        "p05": stats.get("avg_rad_p5"),
        "p10": stats.get("avg_rad_p10"),
        "p25": stats.get("avg_rad_p25"),
        "p50": stats.get("avg_rad_p50"),
        "p75": stats.get("avg_rad_p75"),
        "p90": stats.get("avg_rad_p90"),
        "p95": stats.get("avg_rad_p95"),
        "p99": stats.get("avg_rad_p99")
    })


valid_distribution = ee.FeatureCollection(
    viirs_valid.map(get_image_stats)
)

distribution_data = valid_distribution.getInfo()["features"]

distribution_df = pd.DataFrame([
    feature["properties"]
    for feature in distribution_data
])

distribution_df["date"] = pd.to_datetime(
    distribution_df["date"]
)

distribution_df = distribution_df.sort_values(
    "date"
).reset_index(drop=True)

print(
    "Number of valid months:",
    len(distribution_df)
)

distribution_df.head()

In [ ]:
# ==========================================
# Cell 16 — Global VIIRS Radiance Distribution
# Final Version
# ==========================================

# ------------------------------------------
# Global statistics
# ------------------------------------------

global_stats = radiance_samples.reduceColumns(
    reducer=ee.Reducer.mean()
        .combine(
            reducer2=ee.Reducer.minMax(),
            sharedInputs=True
        )
        .combine(
            reducer2=ee.Reducer.stdDev(),
            sharedInputs=True
        )
        .combine(
            reducer2=ee.Reducer.percentile(
                [1, 5, 10, 25, 50, 75, 90, 95, 99]
            ),
            sharedInputs=True
        ),
    selectors=["avg_rad"]
).getInfo()


# ------------------------------------------
# Print results
# ------------------------------------------

print("========== GLOBAL RADIANCE DISTRIBUTION ==========")

print("Mean :", global_stats["mean"])
print("Min  :", global_stats["min"])
print("Max  :", global_stats["max"])
print("Std  :", global_stats["stdDev"])

print("\n========== PERCENTILES ==========")

for key in [
    "p1",
    "p5",
    "p10",
    "p25",
    "p50",
    "p75",
    "p90",
    "p95",
    "p99"
]:
    print(
        key.upper(),
        ":",
        global_stats[key]
    )

In [ ]:
# ==========================================
# Cell 17 — VIIRS Radiance Histogram
# Corrected Version
# ==========================================

histogram = radiance_samples.reduceColumns(
    reducer=ee.Reducer.fixedHistogram(
        0,
        100,
        100
    ),
    selectors=["avg_rad"]
).getInfo()


# ------------------------------------------
# Convert histogram to DataFrame
# ------------------------------------------

histogram_array = np.array(
    histogram["histogram"]
)

histogram_df = pd.DataFrame(
    histogram_array,
    columns=[
        "lower_bound",
        "count"
    ]
)

# Calculate bin width
bin_width = (
    100 - 0
) / 100

# Calculate upper bound
histogram_df["upper_bound"] = (
    histogram_df["lower_bound"]
    + bin_width
)

# Calculate midpoint
histogram_df["midpoint"] = (
    histogram_df["lower_bound"]
    + histogram_df["upper_bound"]
) / 2


# ------------------------------------------
# Reorder columns
# ------------------------------------------

histogram_df = histogram_df[
    [
        "lower_bound",
        "upper_bound",
        "midpoint",
        "count"
    ]
]


# ------------------------------------------
# Print results
# ------------------------------------------

print("========== HISTOGRAM ==========")

print(
    histogram_df.head(20).to_string(
        index=False
    )
)

print(
    "\nTotal observations in histogram:",
    int(histogram_df["count"].sum())
)

In [ ]:
# ==========================================
# Cell 18 — Candidate Brightness Thresholds
# ==========================================

thresholds = [
    0.5,
    1,
    2,
    3,
    5,
    10,
    20,
    30,
    40,
    50
]

threshold_results = []

for threshold in thresholds:

    count_above = (
        radiance_samples
        .filter(
            ee.Filter.gt(
                "avg_rad",
                threshold
            )
        )
        .size()
        .getInfo()
    )

    percentage = (
        100
        * count_above
        / radiance_samples.size().getInfo()
    )

    threshold_results.append({
        "threshold": threshold,
        "bright_count": count_above,
        "bright_percentage": percentage
    })


threshold_df = pd.DataFrame(
    threshold_results
)


print("========== BRIGHTNESS THRESHOLD ANALYSIS ==========")

print(
    threshold_df.to_string(
        index=False
    )
)

In [ ]:
# ==========================================
# Cell 19 — Temporal Stability of Thresholds
# ==========================================

candidate_thresholds = [
    1,
    2,
    5,
    10
]


def calculate_monthly_bright_percentage(image):

    image = ee.Image(image)

    total = image.select("avg_rad").reduceRegion(
        reducer=ee.Reducer.count(),
        geometry=roi,
        scale=500,
        maxPixels=1e9
    ).get("avg_rad")

    results = {
        "date": ee.Date(
            image.get("system:time_start")
        ).format("YYYY-MM-dd")
    }

    for threshold in candidate_thresholds:

        bright_count = (
            image
            .select("avg_rad")
            .gt(threshold)
            .reduceRegion(
                reducer=ee.Reducer.sum(),
                geometry=roi,
                scale=500,
                maxPixels=1e9
            )
            .get("avg_rad")
        )

        percentage = (
            ee.Number(bright_count)
            .divide(total)
            .multiply(100)
        )

        results[
            f"bright_pct_{threshold}"
        ] = percentage

    return ee.Feature(
        None,
        results
    )


monthly_threshold_stats = ee.FeatureCollection(
    viirs_valid.map(
        calculate_monthly_bright_percentage
    )
)


# ------------------------------------------
# Convert to Pandas
# ------------------------------------------

threshold_data = (
    monthly_threshold_stats
    .getInfo()["features"]
)

threshold_temporal_df = pd.DataFrame([
    feature["properties"]
    for feature in threshold_data
])

threshold_temporal_df["date"] = pd.to_datetime(
    threshold_temporal_df["date"]
)

threshold_temporal_df = threshold_temporal_df.sort_values(
    "date"
).reset_index(drop=True)


# ------------------------------------------
# Summary statistics
# ------------------------------------------

print("========== TEMPORAL THRESHOLD STABILITY ==========")

summary = []

for threshold in candidate_thresholds:

    column = f"bright_pct_{threshold}"

    summary.append({
        "threshold": threshold,
        "mean_bright_pct":
            threshold_temporal_df[column].mean(),
        "std_bright_pct":
            threshold_temporal_df[column].std(),
        "min_bright_pct":
            threshold_temporal_df[column].min(),
        "max_bright_pct":
            threshold_temporal_df[column].max()
    })


threshold_stability_df = pd.DataFrame(summary)

print(
    threshold_stability_df.to_string(
        index=False
    )
)

In [ ]:
# ==========================================
# Cell 20 — Diagnostic: Validity Flag
# ==========================================

def add_validity_flag(image):

    image = ee.Image(image)

    mean_cf = image.select("cf_cvg").reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=500,
        maxPixels=1e9
    ).get("cf_cvg")

    # 1 = valid, 0 = invalid
    validity = ee.Algorithms.If(
        ee.Number(mean_cf).gt(0),
        1,
        0
    )

    return image.set(
        "valid_flag",
        validity
    )


viirs_flagged = viirs_monthly.map(
    add_validity_flag
)


# ------------------------------------------
# Check first 15 images
# ------------------------------------------

flagged_list = (
    viirs_flagged
    .limit(15)
    .getInfo()["features"]
)

for feature in flagged_list:

    date = ee.Date(
        feature["properties"]["system:time_start"]
    ).format("YYYY-MM-dd").getInfo()

    flag = feature["properties"].get(
        "valid_flag"
    )

    print(date, "→ valid_flag =", flag)


# ------------------------------------------
# Count valid / invalid
# ------------------------------------------

valid_months = viirs_flagged.filter(
    ee.Filter.eq("valid_flag", 1)
)

invalid_months = viirs_flagged.filter(
    ee.Filter.eq("valid_flag", 0)
)


print("\nValid months:",
      valid_months.size().getInfo())

print("Invalid months:",
      invalid_months.size().getInfo())

In [ ]:
# ==========================================
# Cell 21 — Robust Temporal Interpolation
# ==========================================

def interpolate_invalid_month(image):

    image = ee.Image(image)

    current_time = ee.Number(
        image.get("system:time_start")
    )

    # --------------------------------------
    # Nearest valid month BEFORE
    # --------------------------------------

    before = (
        valid_months
        .filter(
            ee.Filter.lt(
                "system:time_start",
                current_time
            )
        )
        .sort(
            "system:time_start",
            False
        )
        .first()
    )

    # --------------------------------------
    # Nearest valid month AFTER
    # --------------------------------------

    after = (
        valid_months
        .filter(
            ee.Filter.gt(
                "system:time_start",
                current_time
            )
        )
        .sort(
            "system:time_start",
            True
        )
        .first()
    )

    before = ee.Image(before)
    after = ee.Image(after)

    # --------------------------------------
    # Radiance
    # --------------------------------------

    before_rad = before.select("avg_rad")
    after_rad = after.select("avg_rad")

    # --------------------------------------
    # Time
    # --------------------------------------

    before_time = ee.Number(
        before.get("system:time_start")
    )

    after_time = ee.Number(
        after.get("system:time_start")
    )

    # --------------------------------------
    # Interpolation fraction
    # --------------------------------------

    fraction = (
        current_time
        .subtract(before_time)
        .divide(
            after_time.subtract(before_time)
        )
    )

    # --------------------------------------
    # Linear interpolation
    # --------------------------------------

    interpolated = (
        before_rad
        .add(
            after_rad
            .subtract(before_rad)
            .multiply(fraction)
        )
        .rename("avg_rad")
    )

    return (
        interpolated
        .copyProperties(
            image,
            ["system:time_start"]
        )
        .set(
            "interpolated",
            True
        )
    )


# ------------------------------------------
# Interpolate the 12 invalid months
# ------------------------------------------

interpolated_invalid = invalid_months.map(
    interpolate_invalid_month
)


# ------------------------------------------
# Keep the 120 valid months unchanged
# ------------------------------------------

def mark_valid_image(image):

    return (
        ee.Image(image)
        .select("avg_rad")
        .copyProperties(
            image,
            ["system:time_start"]
        )
        .set(
            "interpolated",
            False
        )
    )


valid_output = valid_months.map(
    mark_valid_image
)


# ------------------------------------------
# Final 132-month collection
# ------------------------------------------

viirs_interpolated = (
    valid_output
    .merge(interpolated_invalid)
    .sort("system:time_start")
)


print(
    "Valid original months:",
    valid_output.size().getInfo()
)

print(
    "Interpolated months:",
    interpolated_invalid.size().getInfo()
)

print(
    "Final temporal series:",
    viirs_interpolated.size().getInfo()
)

In [ ]:
# ==========================================
# Cell 22 — Persistence Frequency
# ==========================================

candidate_thresholds = [2, 5, 10]

persistence_frequency = {}

total_months = viirs_interpolated.size()

print("Total temporal observations:",
      total_months.getInfo())


for threshold in candidate_thresholds:

    # Identify bright pixels in each month
    bright_collection = viirs_interpolated.map(
        lambda image:
            ee.Image(image)
            .gt(threshold)
            .rename("bright")
    )

    # Number of bright months for each pixel
    bright_count = bright_collection.sum()

    # Persistence frequency
    frequency = (
        bright_count
        .divide(total_months)
        .rename(
            f"Persistence_Frequency_{threshold}"
        )
    )

    persistence_frequency[threshold] = frequency


print("\nPersistence frequency maps created:")

for threshold in candidate_thresholds:

    print(
        threshold,
        "→",
        persistence_frequency[threshold]
        .bandNames()
        .getInfo()
    )

In [ ]:
# ==========================================
# Cell 23 — Spatial Distribution of
# Persistence Frequency
# ==========================================

frequency_levels = [0.50, 0.75, 0.90]

frequency_results = []


for threshold in candidate_thresholds:

    frequency_image = persistence_frequency[
        threshold
    ]

    for level in frequency_levels:

        persistent_area = (
            frequency_image
            .gt(level)
            .selfMask()
            .multiply(
                ee.Image.pixelArea()
            )
            .reduceRegion(
                reducer=ee.Reducer.sum(),
                geometry=roi,
                scale=500,
                maxPixels=1e10
            )
            .get(
                f"Persistence_Frequency_{threshold}"
            )
        )

        area_km2 = (
            ee.Number(persistent_area)
            .divide(1e6)
        )

        frequency_results.append({
            "threshold": threshold,
            "frequency_level": level,
            "area_km2": area_km2
        })


# ------------------------------------------
# Convert to FeatureCollection
# ------------------------------------------

frequency_results_fc = ee.FeatureCollection([
    ee.Feature(
        None,
        result
    )
    for result in frequency_results
])


# ------------------------------------------
# Get results
# ------------------------------------------

frequency_results_data = (
    frequency_results_fc
    .getInfo()["features"]
)

frequency_results_df = pd.DataFrame([
    feature["properties"]
    for feature in frequency_results_data
])


print(
    frequency_results_df
    .sort_values(
        ["threshold", "frequency_level"]
    )
    .to_string(index=False)
)

In [ ]:
# ==========================================
# Cell 24 — Visualize Persistence Frequency
# ==========================================

Map = geemap.Map(basemap="SATELLITE")
Map.centerObject(roi, 8)

# ------------------------------------------
# Visualization parameters
# ------------------------------------------

vis_params = {
    "min": 0,
    "max": 1,
    "palette": [
        "000000",
        "440154",
        "31688E",
        "35B779",
        "FDE725"
    ]
}


# ------------------------------------------
# Add frequency maps
# ------------------------------------------

Map.addLayer(
    persistence_frequency[2].clip(roi),
    vis_params,
    "Persistence Frequency - Threshold 2"
)

Map.addLayer(
    persistence_frequency[5].clip(roi),
    vis_params,
    "Persistence Frequency - Threshold 5"
)

Map.addLayer(
    persistence_frequency[10].clip(roi),
    vis_params,
    "Persistence Frequency - Threshold 10"
)


# ------------------------------------------
# ROI boundary
# ------------------------------------------

Map.addLayer(
    tehran.style(
        color="red",
        fillColor="00000000",
        width=2
    ),
    {},
    "Tehran Province"
)

Map.add_layer_control()

Map

In [ ]:
# ==========================================
# Cell 24 — Compare Persistence Maps Separately
# ==========================================

for threshold in [2, 5, 10]:

    Map = geemap.Map(basemap="SATELLITE")
    Map.centerObject(roi, 8)

    Map.addLayer(
        persistence_frequency[threshold].clip(roi),
        {
            "min": 0,
            "max": 1,
            "palette": [
                "000000",
                "440154",
                "31688E",
                "35B779",
                "FDE725"
            ]
        },
        f"Persistence Frequency - Threshold {threshold}"
    )

    Map.addLayer(
        tehran.style(
            color="red",
            fillColor="00000000",
            width=2
        ),
        {},
        "Tehran Province"
    )

    Map.addLayerControl()

    print(f"Threshold = {threshold}")
    display(Map)

In [ ]:
# ==========================================
# Cell 25 — Persistence Frequency
# Based on Valid Observed Months
# ==========================================

candidate_thresholds = [2, 5, 10]

persistence_frequency_valid = {}

total_valid_months = valid_months.size()

for threshold in candidate_thresholds:

    bright_collection = valid_months.map(
        lambda image:
            ee.Image(image)
            .select("avg_rad")
            .gt(threshold)
            .rename("bright")
    )

    bright_count = bright_collection.sum()

    frequency = (
        bright_count
        .divide(total_valid_months)
        .rename(f"Persistence_Frequency_{threshold}")
    )

    persistence_frequency_valid[threshold] = frequency


print(
    "Number of valid observed months:",
    total_valid_months.getInfo()
)

print("\nPersistence frequency maps created:")

for threshold in candidate_thresholds:
    print(
        threshold,
        "→",
        persistence_frequency_valid[threshold].bandNames().getInfo()
    )

In [ ]:
# ==========================================
# Cell 26 — Temporal Nighttime Light Intensity
# Mean Radiance During Bright Observations
# ==========================================

intensity_maps = {}

for threshold in candidate_thresholds:

    def get_bright_radiance(image):

        image = ee.Image(image)

        radiance = image.select("avg_rad")

        # Keep only observations above threshold
        bright_radiance = radiance.updateMask(
            radiance.gt(threshold)
        )

        return bright_radiance.rename("bright_radiance")


    bright_radiance_collection = valid_months.map(
        get_bright_radiance
    )


    # Sum of radiance values during bright months
    radiance_sum = bright_radiance_collection.sum()


    # Number of bright observations
    bright_count = (
        bright_radiance_collection
        .map(
            lambda image:
                image.mask().rename("bright_count")
        )
        .sum()
    )


    # Mean radiance during bright observations
    intensity = (
        radiance_sum
        .divide(bright_count)
        .rename(f"Intensity_{threshold}")
    )


    intensity_maps[threshold] = intensity


print("Intensity maps created:")

for threshold in candidate_thresholds:
    print(
        threshold,
        "→",
        intensity_maps[threshold].bandNames().getInfo()
    )

In [ ]:
# ==========================================
# Cell 27 — Intensity Distribution Statistics
# ==========================================

intensity_stats = {}

for threshold in candidate_thresholds:

    intensity_image = intensity_maps[threshold]

    stats = intensity_image.reduceRegion(
        reducer=(
            ee.Reducer.minMax()
            .combine(
                reducer2=ee.Reducer.mean(),
                sharedInputs=True
            )
            .combine(
                reducer2=ee.Reducer.stdDev(),
                sharedInputs=True
            )
            .combine(
                reducer2=ee.Reducer.percentile(
                    [25, 50, 75, 90, 95, 99]
                ),
                sharedInputs=True
            )
        ),
        geometry=roi,
        scale=500,
        maxPixels=1e9
    )

    intensity_stats[threshold] = stats.getInfo()


for threshold in candidate_thresholds:

    print(f"\nThreshold = {threshold}")

    stats = intensity_stats[threshold]

    for key, value in stats.items():
        print(f"{key}: {value}")

In [ ]:
# ==========================================
# Cell 28 — Robust Intensity Normalization
# ==========================================

intensity_normalized = {}

# P95 values obtained from Cell 27
p95_values = {
    2: intensity_stats[2]["Intensity_2_p95"],
    5: intensity_stats[5]["Intensity_5_p95"],
    10: intensity_stats[10]["Intensity_10_p95"]
}

for threshold in candidate_thresholds:

    intensity_image = intensity_maps[threshold]

    p95 = ee.Number(p95_values[threshold])

    # Normalize intensity by P95
    normalized = (
        intensity_image
        .divide(p95)
        .min(1)
        .rename(f"Intensity_Normalized_{threshold}")
    )

    intensity_normalized[threshold] = normalized


print("Normalized intensity maps created:")

for threshold in candidate_thresholds:
    print(
        threshold,
        "→",
        intensity_normalized[threshold]
        .bandNames()
        .getInfo()
    )

In [ ]:
# ==========================================
# Cell 29 — Temporal Stability
# Coefficient of Variation (CV)
# ==========================================

stability_maps = {}
cv_maps = {}

for threshold in candidate_thresholds:

    def bright_radiance_for_stability(image):

        image = ee.Image(image)

        radiance = image.select("avg_rad")

        # Only bright observations
        return radiance.updateMask(
            radiance.gt(threshold)
        ).rename("bright_radiance")


    bright_collection = valid_months.map(
        bright_radiance_for_stability
    )


    # Mean radiance during bright observations
    mean_radiance = (
        bright_collection
        .mean()
        .rename("mean_radiance")
    )


    # Standard deviation during bright observations
    std_radiance = (
        bright_collection
        .reduce(ee.Reducer.stdDev())
        .rename("std_radiance")
    )


    # Coefficient of Variation
    cv = (
        std_radiance
        .divide(mean_radiance)
        .rename(f"CV_{threshold}")
    )


    # Stability score
    stability = (
        ee.Image(1)
        .divide(
            ee.Image(1).add(cv)
        )
        .rename(f"Stability_{threshold}")
    )


    cv_maps[threshold] = cv
    stability_maps[threshold] = stability


print("Stability maps created:")

for threshold in candidate_thresholds:

    print(
        threshold,
        "→",
        stability_maps[threshold]
        .bandNames()
        .getInfo()
    )

In [ ]:
# ==========================================
# Cell 30 — Stability Distribution
# ==========================================

stability_stats = {}

for threshold in candidate_thresholds:

    stability_image = stability_maps[threshold]

    stats = stability_image.reduceRegion(
        reducer=(
            ee.Reducer.minMax()
            .combine(
                reducer2=ee.Reducer.mean(),
                sharedInputs=True
            )
            .combine(
                reducer2=ee.Reducer.stdDev(),
                sharedInputs=True
            )
            .combine(
                reducer2=ee.Reducer.percentile(
                    [25, 50, 75, 90, 95, 99]
                ),
                sharedInputs=True
            )
        ),
        geometry=roi,
        scale=500,
        maxPixels=1e9
    )

    stability_stats[threshold] = stats.getInfo()


for threshold in candidate_thresholds:

    print(f"\nThreshold = {threshold}")

    stats = stability_stats[threshold]

    for key, value in stats.items():
        print(f"{key}: {value}")

In [ ]:
# ==========================================
# Cell 31 — Bright Observation Count
# ==========================================

bright_count_maps = {}

for threshold in candidate_thresholds:

    bright_collection = valid_months.map(
        lambda image:
            ee.Image(image)
            .select("avg_rad")
            .gt(threshold)
            .rename("bright")
    )

    bright_count = (
        bright_collection
        .sum()
        .rename(f"Bright_Count_{threshold}")
    )

    bright_count_maps[threshold] = bright_count


print("Bright observation count maps created:")

for threshold in candidate_thresholds:

    print(
        threshold,
        "→",
        bright_count_maps[threshold]
        .bandNames()
        .getInfo()
    )

In [ ]:
# ==========================================
# Cell 32 — Bright Observation Count Statistics
# ==========================================

bright_count_stats = {}

for threshold in candidate_thresholds:

    count_image = bright_count_maps[threshold]

    stats = count_image.reduceRegion(
        reducer=(
            ee.Reducer.minMax()
            .combine(
                reducer2=ee.Reducer.mean(),
                sharedInputs=True
            )
            .combine(
                reducer2=ee.Reducer.stdDev(),
                sharedInputs=True
            )
            .combine(
                reducer2=ee.Reducer.percentile(
                    [10, 25, 50, 75, 90, 95, 99]
                ),
                sharedInputs=True
            )
        ),
        geometry=roi,
        scale=500,
        maxPixels=1e9
    )

    bright_count_stats[threshold] = stats.getInfo()


for threshold in candidate_thresholds:

    print(f"\nThreshold = {threshold}")

    stats = bright_count_stats[threshold]

    for key, value in stats.items():
        print(f"{key}: {value}")

In [ ]:
# ==========================================
# Cell 33 — Frequency vs Stability by Bins
# ==========================================

frequency_bins = [
    (0, 5),
    (6, 20),
    (21, 40),
    (41, 60),
    (61, 80),
    (81, 100),
    (101, 120)
]

frequency_stability_table = []

for threshold in candidate_thresholds:

    count_image = bright_count_maps[threshold]
    stability_image = stability_maps[threshold]

    print(f"\nThreshold = {threshold}")

    for lower, upper in frequency_bins:

        mask = (
            count_image.gte(lower)
            .And(count_image.lte(upper))
        )

        mean_stability = (
            stability_image
            .updateMask(mask)
            .reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=roi,
                scale=500,
                maxPixels=1e9
            )
            .get(f"Stability_{threshold}")
        )

        pixel_count = (
            mask
            .reduceRegion(
                reducer=ee.Reducer.sum(),
                geometry=roi,
                scale=500,
                maxPixels=1e9
            )
            .get(
                f"Bright_Count_{threshold}"
            )
        )

        frequency_stability_table.append({
            "threshold": threshold,
            "frequency_range": f"{lower}-{upper}",
            "pixel_count": pixel_count,
            "mean_stability": mean_stability
        })


frequency_stability_fc = ee.FeatureCollection([
    ee.Feature(None, item)
    for item in frequency_stability_table
])

frequency_stability_data = (
    frequency_stability_fc
    .getInfo()["features"]
)

frequency_stability_df = pd.DataFrame([
    feature["properties"]
    for feature in frequency_stability_data
])

print(
    frequency_stability_df.to_string(
        index=False
    )
)

In [ ]:
# ==========================================
# Cell 34 — Frequency Distribution
# Distribution of temporal brightness frequency
# ==========================================

frequency_bins_percent = [
    (0, 10),
    (10, 25),
    (25, 50),
    (50, 75),
    (75, 90),
    (90, 100)
]

frequency_distribution = []

for threshold in candidate_thresholds:

    frequency_image = persistence_frequency_valid[threshold]

    print(f"\nThreshold = {threshold}")

    for lower, upper in frequency_bins_percent:

        mask = (
            frequency_image.gte(lower / 100)
            .And(frequency_image.lt(upper / 100))
        )

        pixel_count = (
            mask
            .reduceRegion(
                reducer=ee.Reducer.sum(),
                geometry=roi,
                scale=500,
                maxPixels=1e9
            )
            .get(
                f"Persistence_Frequency_{threshold}"
            )
        )

        frequency_distribution.append({
            "threshold": threshold,
            "frequency_range": f"{lower}-{upper}%",
            "pixel_count": pixel_count
        })

frequency_distribution_fc = ee.FeatureCollection([
    ee.Feature(None, item)
    for item in frequency_distribution
])

frequency_distribution_data = (
    frequency_distribution_fc
    .getInfo()["features"]
)

frequency_distribution_df = pd.DataFrame([
    feature["properties"]
    for feature in frequency_distribution_data
])

print(
    frequency_distribution_df.to_string(
        index=False
    )
)

In [ ]:
# ==========================================
# Cell 35 — Detailed Frequency Analysis
# ==========================================

frequency_analysis = []

for threshold in candidate_thresholds:

    frequency_image = persistence_frequency_valid[threshold]

    print(f"\n{'=' * 50}")
    print(f"Threshold = {threshold}")
    print(f"{'=' * 50}")

    # ------------------------------------------
    # 1. Frequency percentiles
    # ------------------------------------------

    frequency_percentiles = frequency_image.reduceRegion(
        reducer=ee.Reducer.percentile(
            [25, 50, 75, 90, 95, 99]
        ),
        geometry=roi,
        scale=500,
        maxPixels=1e9
    )

    percentile_values = frequency_percentiles.getInfo()

    print("\nFrequency percentiles:")

    for p in [25, 50, 75, 90, 95, 99]:
        key = f"Persistence_Frequency_{threshold}_p{p}"
        value = percentile_values.get(key)
        print(f"P{p}: {value:.4f}")


    # ------------------------------------------
    # 2. Area above frequency thresholds
    # ------------------------------------------

    frequency_cutoffs = [0.50, 0.75, 0.90, 0.95, 1.00]

    print("\nArea above frequency thresholds:")

    for cutoff in frequency_cutoffs:

        if cutoff < 1.0:
            mask = frequency_image.gte(cutoff)
        else:
            mask = frequency_image.eq(1)

        area_km2 = (
            ee.Image.pixelArea()
            .updateMask(mask)
            .reduceRegion(
                reducer=ee.Reducer.sum(),
                geometry=roi,
                scale=500,
                maxPixels=1e9
            )
            .get("area")
        )

        area_km2 = ee.Number(area_km2).divide(1e6).getInfo()

        print(
            f"Frequency >= {cutoff * 100:.0f}%: "
            f"{area_km2:.2f} km²"
        )

        frequency_analysis.append({
            "threshold": threshold,
            "frequency_cutoff": cutoff,
            "area_km2": area_km2
        })

In [ ]:
# ==========================================
# Cell 36 — Spatial Agreement Between Thresholds
# ==========================================

frequency_cutoffs = [0.50, 0.75, 0.90]

threshold_pairs = [
    (2, 5),
    (2, 10),
    (5, 10)
]

agreement_results = []

for cutoff in frequency_cutoffs:

    print(f"\n{'=' * 60}")
    print(f"Frequency >= {cutoff * 100:.0f}%")
    print(f"{'=' * 60}")

    masks = {}

    # ------------------------------------------
    # Create persistent masks for each threshold
    # ------------------------------------------

    for threshold in candidate_thresholds:

        frequency_image = persistence_frequency_valid[threshold]

        masks[threshold] = frequency_image.gte(cutoff)


    # ------------------------------------------
    # Pairwise spatial agreement
    # ------------------------------------------

    for t1, t2 in threshold_pairs:

        mask1 = masks[t1]
        mask2 = masks[t2]

        intersection = mask1.And(mask2)
        union = mask1.Or(mask2)

        intersection_area = (
            ee.Image.pixelArea()
            .updateMask(intersection)
            .reduceRegion(
                reducer=ee.Reducer.sum(),
                geometry=roi,
                scale=500,
                maxPixels=1e9
            )
            .get("area")
        )

        union_area = (
            ee.Image.pixelArea()
            .updateMask(union)
            .reduceRegion(
                reducer=ee.Reducer.sum(),
                geometry=roi,
                scale=500,
                maxPixels=1e9
            )
            .get("area")
        )

        intersection_area = (
            ee.Number(intersection_area)
            .divide(1e6)
            .getInfo()
        )

        union_area = (
            ee.Number(union_area)
            .divide(1e6)
            .getInfo()
        )

        # Jaccard similarity
        jaccard = (
            intersection_area / union_area
            if union_area > 0
            else np.nan
        )

        agreement_results.append({
            "frequency_cutoff": cutoff,
            "threshold_1": t1,
            "threshold_2": t2,
            "intersection_km2": intersection_area,
            "union_km2": union_area,
            "jaccard": jaccard
        })

        print(
            f"Threshold {t1} vs {t2} | "
            f"Intersection = {intersection_area:.2f} km² | "
            f"Union = {union_area:.2f} km² | "
            f"Jaccard = {jaccard:.3f}"
        )


# ------------------------------------------
# Final table
# ------------------------------------------

agreement_df = pd.DataFrame(agreement_results)

print("\n")
print(
    agreement_df.to_string(
        index=False
    )
)

In [ ]:
# ==========================================
# Cell 37 — Threshold Retention Analysis
# ==========================================

frequency_cutoffs = [0.50, 0.75, 0.90]

retention_results = []

for cutoff in frequency_cutoffs:

    print(f"\n{'=' * 60}")
    print(f"Frequency >= {cutoff * 100:.0f}%")
    print(f"{'=' * 60}")

    areas = {}

    # ------------------------------------------
    # Calculate area for each radiance threshold
    # ------------------------------------------

    for threshold in candidate_thresholds:

        frequency_image = persistence_frequency_valid[threshold]

        mask = frequency_image.gte(cutoff)

        area_km2 = (
            ee.Image.pixelArea()
            .updateMask(mask)
            .reduceRegion(
                reducer=ee.Reducer.sum(),
                geometry=roi,
                scale=500,
                maxPixels=1e9
            )
            .get("area")
        )

        area_km2 = (
            ee.Number(area_km2)
            .divide(1e6)
            .getInfo()
        )

        areas[threshold] = area_km2

    # ------------------------------------------
    # Calculate retention relative to Threshold=2
    # ------------------------------------------

    base_area = areas[2]

    for threshold in candidate_thresholds:

        retention = (
            (areas[threshold] / base_area) * 100
            if base_area > 0
            else np.nan
        )

        retention_results.append({
            "frequency_cutoff": cutoff,
            "threshold": threshold,
            "area_km2": areas[threshold],
            "retention_vs_threshold_2_percent": retention
        })

        print(
            f"Threshold {threshold}: "
            f"Area = {areas[threshold]:.2f} km² | "
            f"Retention vs T=2 = {retention:.2f}%"
        )


# ------------------------------------------
# Final table
# ------------------------------------------

retention_df = pd.DataFrame(retention_results)

print("\n")
print(
    retention_df.to_string(
        index=False
    )
)

In [ ]:
# ==========================================
# Cell 38 — Relationship Between
# Frequency, Intensity and Stability
# ==========================================

correlation_results = []

for threshold in candidate_thresholds:

    frequency = persistence_frequency_valid[threshold]
    intensity = intensity_maps[threshold]
    stability = stability_maps[threshold]

    print(f"\n{'=' * 60}")
    print(f"Threshold = {threshold}")
    print(f"{'=' * 60}")

    # ------------------------------------------
    # Frequency vs Intensity
    # ------------------------------------------

    freq_intensity = (
        frequency.addBands(intensity)
        .reduceRegion(
            reducer=ee.Reducer.pearsonsCorrelation(),
            geometry=roi,
            scale=500,
            maxPixels=1e9
        )
    )

    corr_fi = freq_intensity.getInfo()

    # ------------------------------------------
    # Frequency vs Stability
    # ------------------------------------------

    freq_stability = (
        frequency.addBands(stability)
        .reduceRegion(
            reducer=ee.Reducer.pearsonsCorrelation(),
            geometry=roi,
            scale=500,
            maxPixels=1e9
        )
    )

    corr_fs = freq_stability.getInfo()

    print(
        "Frequency vs Intensity:",
        corr_fi
    )

    print(
        "Frequency vs Stability:",
        corr_fs
    )

    correlation_results.append({
        "threshold": threshold,
        "frequency_intensity_correlation": list(
            corr_fi.values()
        )[0],
        "frequency_stability_correlation": list(
            corr_fs.values()
        )[0]
    })


correlation_df = pd.DataFrame(
    correlation_results
)

print("\nCorrelation summary:")
print(
    correlation_df.to_string(
        index=False
    )
)

In [ ]:
# ==========================================
# Cell 39 — Candidate Persistence Formulations
# ==========================================

candidate_persistence_results = []

for threshold in candidate_thresholds:

    # ------------------------------------------
    # Components
    # ------------------------------------------

    F = persistence_frequency_valid[threshold]

    I = intensity_normalized[threshold]

    S = stability_maps[threshold]

    # ------------------------------------------
    # Candidate A: Arithmetic Mean
    # ------------------------------------------

    P_arithmetic = (
        F.add(I).add(S)
        .divide(3)
        .rename(f"P_Arithmetic_{threshold}")
    )

    # ------------------------------------------
    # Candidate B: Geometric Mean
    # ------------------------------------------

    P_geometric = (
        F.multiply(I).multiply(S)
        .pow(ee.Number(1).divide(3))
        .rename(f"P_Geometric_{threshold}")
    )

    # ------------------------------------------
    # Candidate C: Multiplicative
    # ------------------------------------------

    P_product = (
        F.multiply(I).multiply(S)
        .rename(f"P_Product_{threshold}")
    )

    candidate_maps = {
        "Arithmetic": P_arithmetic,
        "Geometric": P_geometric,
        "Product": P_product
    }

    print(f"\n{'=' * 65}")
    print(f"Threshold = {threshold}")
    print(f"{'=' * 65}")

    # ------------------------------------------
    # Statistics for each candidate
    # ------------------------------------------

    for name, image in candidate_maps.items():

        stats = image.reduceRegion(
            reducer=ee.Reducer.percentile(
                [1, 5, 25, 50, 75, 90, 95, 99]
            ),
            geometry=roi,
            scale=500,
            maxPixels=1e9
        ).getInfo()

        print(f"\n{name} Persistence:")

        for p in [1, 5, 25, 50, 75, 90, 95, 99]:

            key = f"P_{name}_{threshold}_p{p}"
            value = stats.get(key)

            print(
                f"P{p}: {value:.4f}"
            )

        candidate_persistence_results.append({
            "threshold": threshold,
            "method": name,
            "P1": stats.get(
                f"P_{name}_{threshold}_p1"
            ),
            "P5": stats.get(
                f"P_{name}_{threshold}_p5"
            ),
            "P25": stats.get(
                f"P_{name}_{threshold}_p25"
            ),
            "P50": stats.get(
                f"P_{name}_{threshold}_p50"
            ),
            "P75": stats.get(
                f"P_{name}_{threshold}_p75"
            ),
            "P90": stats.get(
                f"P_{name}_{threshold}_p90"
            ),
            "P95": stats.get(
                f"P_{name}_{threshold}_p95"
            ),
            "P99": stats.get(
                f"P_{name}_{threshold}_p99"
            )
        })


candidate_persistence_df = pd.DataFrame(
    candidate_persistence_results
)

print("\n")
print(
    candidate_persistence_df.to_string(
        index=False
    )
)

In [ ]:
# ==========================================
# Cell 40 — Persistence Behavior by Frequency
# ==========================================

frequency_bins = [
    (0.00, 0.10),
    (0.10, 0.25),
    (0.25, 0.50),
    (0.50, 0.75),
    (0.75, 0.90),
    (0.90, 1.01)
]

persistence_frequency_behavior = []

for threshold in candidate_thresholds:

    F = persistence_frequency_valid[threshold]
    I = intensity_normalized[threshold]
    S = stability_maps[threshold]

    P_arithmetic = (
        F.add(I).add(S)
        .divide(3)
        .rename("P_Arithmetic")
    )

    P_product = (
        F.multiply(I).multiply(S)
        .rename("P_Product")
    )

    print(f"\n{'=' * 70}")
    print(f"Threshold = {threshold}")
    print(f"{'=' * 70}")

    for lower, upper in frequency_bins:

        frequency_mask = (
            F.gte(lower)
            .And(F.lt(upper))
        )

        arithmetic_mean = (
            P_arithmetic
            .updateMask(frequency_mask)
            .reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=roi,
                scale=500,
                maxPixels=1e9
            )
            .get("P_Arithmetic")
        )

        product_mean = (
            P_product
            .updateMask(frequency_mask)
            .reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=roi,
                scale=500,
                maxPixels=1e9
            )
            .get("P_Product")
        )

        pixel_count = (
            frequency_mask
            .reduceRegion(
                reducer=ee.Reducer.sum(),
                geometry=roi,
                scale=500,
                maxPixels=1e9
            )
            .get(
                f"Persistence_Frequency_{threshold}"
            )
        )

        arithmetic_mean = (
            ee.Number(arithmetic_mean).getInfo()
        )

        product_mean = (
            ee.Number(product_mean).getInfo()
        )

        pixel_count = (
            ee.Number(pixel_count).getInfo()
        )

        print(
            f"Frequency {lower*100:.0f}-{min(upper,1)*100:.0f}% | "
            f"Pixels = {pixel_count:.0f} | "
            f"Arithmetic = {arithmetic_mean:.4f} | "
            f"Product = {product_mean:.4f}"
        )

        persistence_frequency_behavior.append({
            "threshold": threshold,
            "frequency_range": (
                f"{lower*100:.0f}-{min(upper,1)*100:.0f}%"
            ),
            "pixel_count": pixel_count,
            "arithmetic_mean": arithmetic_mean,
            "product_mean": product_mean
        })


persistence_frequency_behavior_df = pd.DataFrame(
    persistence_frequency_behavior
)

print("\n")
print(
    persistence_frequency_behavior_df.to_string(
        index=False
    )
)

In [ ]:
# ==========================================
# Cell 41 — Visualization of Persistence Components
# Threshold = 5
# ==========================================

visual_threshold = 5

F = persistence_frequency_valid[visual_threshold]
I = intensity_normalized[visual_threshold]
S = stability_maps[visual_threshold]

# ------------------------------------------
# Candidate Persistence maps
# ------------------------------------------

P_arithmetic = (
    F.add(I).add(S)
    .divide(3)
    .rename("Persistence_Arithmetic")
)

P_geometric = (
    F.multiply(I).multiply(S)
    .pow(ee.Number(1).divide(3))
    .rename("Persistence_Geometric")
)

# ------------------------------------------
# Create map
# ------------------------------------------

Map_persistence = geemap.Map(
    basemap="SATELLITE"
)

Map_persistence.centerObject(
    roi,
    8
)

# ------------------------------------------
# Frequency
# ------------------------------------------

Map_persistence.addLayer(
    F,
    {
        "min": 0,
        "max": 1,
        "palette": [
            "000000",
            "ffff00",
            "ff0000"
        ]
    },
    "Frequency"
)

# ------------------------------------------
# Normalized Intensity
# ------------------------------------------

Map_persistence.addLayer(
    I,
    {
        "min": 0,
        "max": 1,
        "palette": [
            "000000",
            "00ffff",
            "ffffff"
        ]
    },
    "Normalized Intensity"
)

# ------------------------------------------
# Stability
# ------------------------------------------

Map_persistence.addLayer(
    S,
    {
        "min": 0,
        "max": 1,
        "palette": [
            "000000",
            "00ff00",
            "ffffff"
        ]
    },
    "Stability"
)

# ------------------------------------------
# Arithmetic Persistence
# ------------------------------------------

Map_persistence.addLayer(
    P_arithmetic,
    {
        "min": 0,
        "max": 1,
        "palette": [
            "000000",
            "ffff00",
            "ff0000"
        ]
    },
    "Persistence - Arithmetic"
)

# ------------------------------------------
# Geometric Persistence
# ------------------------------------------

Map_persistence.addLayer(
    P_geometric,
    {
        "min": 0,
        "max": 1,
        "palette": [
            "000000",
            "00ffff",
            "ffffff"
        ]
    },
    "Persistence - Geometric"
)

Map_persistence.addLayer(
    tehran.style(
        color="red",
        fillColor="00000000",
        width=2
    ),
    {},
    "Tehran Province"
)

Map_persistence.add_layer_control()

Map_persistence

In [ ]:
# ==========================================
# Cell 41 — Clean Visualization
# Threshold = 5
# ==========================================

visual_threshold = 5

F = persistence_frequency_valid[visual_threshold]
I = intensity_normalized[visual_threshold]
S = stability_maps[visual_threshold]

P_arithmetic = (
    F.add(I).add(S)
    .divide(3)
    .rename("Persistence_Arithmetic")
)

P_geometric = (
    F.multiply(I).multiply(S)
    .pow(ee.Number(1).divide(3))
    .rename("Persistence_Geometric")
)

Map_persistence = geemap.Map(
    basemap="SATELLITE"
)

Map_persistence.centerObject(
    roi,
    8
)

# ------------------------------------------
# Frequency
# ------------------------------------------

Map_persistence.addLayer(
    F,
    {
        "min": 0,
        "max": 1,
        "palette": [
            "000000",
            "ffff00",
            "ff0000"
        ]
    },
    "1 - Frequency",
    False
)

# ------------------------------------------
# Normalized Intensity
# ------------------------------------------

Map_persistence.addLayer(
    I,
    {
        "min": 0,
        "max": 1,
        "palette": [
            "000000",
            "00ffff",
            "ffffff"
        ]
    },
    "2 - Normalized Intensity",
    False
)

# ------------------------------------------
# Stability
# ------------------------------------------

Map_persistence.addLayer(
    S,
    {
        "min": 0,
        "max": 1,
        "palette": [
            "000000",
            "00ff00",
            "ffffff"
        ]
    },
    "3 - Stability",
    False
)

# ------------------------------------------
# Arithmetic Persistence
# ------------------------------------------

Map_persistence.addLayer(
    P_arithmetic,
    {
        "min": 0,
        "max": 1,
        "palette": [
            "000000",
            "ffff00",
            "ff0000"
        ]
    },
    "4 - Persistence Arithmetic",
    True
)

# ------------------------------------------
# Geometric Persistence
# ------------------------------------------

Map_persistence.addLayer(
    P_geometric,
    {
        "min": 0,
        "max": 1,
        "palette": [
            "000000",
            "00ffff",
            "ffffff"
        ]
    },
    "5 - Persistence Geometric",
    False
)

# ------------------------------------------
# Tehran Province boundary
# ------------------------------------------

Map_persistence.addLayer(
    tehran.style(
        color="red",
        fillColor="00000000",
        width=2
    ),
    {},
    "Tehran Province Boundary",
    True
)

Map_persistence.add_layer_control()

Map_persistence

In [ ]:
# ==========================================
# Cell 42 — Persistence Threshold Sensitivity
# ==========================================

persistence_geometric_maps = {}
persistence_sensitivity_results = []

persistence_cutoffs = [0.50, 0.70, 0.90]

for threshold in candidate_thresholds:

    # ------------------------------------------
    # Components
    # ------------------------------------------

    F = persistence_frequency_valid[threshold]
    I = intensity_normalized[threshold]
    S = stability_maps[threshold]

    # ------------------------------------------
    # Geometric Persistence
    # ------------------------------------------

    P_geometric = (
        F
        .multiply(I)
        .multiply(S)
        .pow(
            ee.Number(1).divide(3)
        )
        .rename(
            f"Persistence_Geometric_{threshold}"
        )
    )

    persistence_geometric_maps[threshold] = (
        P_geometric
    )

    # ------------------------------------------
    # Percentile statistics
    # ------------------------------------------

    percentile_values = (
        P_geometric
        .reduceRegion(
            reducer=ee.Reducer.percentile(
                [1, 5, 25, 50, 75, 90, 95, 99]
            ),
            geometry=roi,
            scale=500,
            maxPixels=1e9
        )
        .getInfo()
    )

    print(
        f"\n{'=' * 70}"
    )
    print(
        f"Threshold = {threshold}"
    )
    print(
        f"{'=' * 70}"
    )

    print("\nGeometric Persistence percentiles:")

    for p in [1, 5, 25, 50, 75, 90, 95, 99]:

        key = (
            f"Persistence_Geometric_{threshold}_p{p}"
        )

        value = percentile_values.get(key)

        print(
            f"P{p}: {value:.4f}"
        )

    # ------------------------------------------
    # Area sensitivity
    # ------------------------------------------

    print("\nArea above Persistence cutoffs:")

    for cutoff in persistence_cutoffs:

        mask = P_geometric.gte(cutoff)

        area_km2 = (
            ee.Image.pixelArea()
            .updateMask(mask)
            .reduceRegion(
                reducer=ee.Reducer.sum(),
                geometry=roi,
                scale=500,
                maxPixels=1e9
            )
            .get("area")
        )

        area_km2 = (
            ee.Number(area_km2)
            .divide(1e6)
            .getInfo()
        )

        print(
            f"Persistence >= {cutoff:.2f}: "
            f"{area_km2:.2f} km²"
        )

        persistence_sensitivity_results.append({
            "radiance_threshold": threshold,
            "persistence_cutoff": cutoff,
            "area_km2": area_km2
        })


persistence_sensitivity_df = pd.DataFrame(
    persistence_sensitivity_results
)

print("\n")
print(
    persistence_sensitivity_df.to_string(
        index=False
    )
)

In [ ]:
# ==========================================
# Cell 43 — Spatial Robustness of Persistence
# ==========================================

persistence_cutoffs_for_comparison = [0.50, 0.70, 0.90]

persistence_robustness_results = []

for cutoff in persistence_cutoffs_for_comparison:

    print("\n")
    print("=" * 70)
    print(f"Persistence >= {cutoff:.0%}")
    print("=" * 70)

    for i in range(len(candidate_thresholds)):

        for j in range(i + 1, len(candidate_thresholds)):

            t1 = candidate_thresholds[i]
            t2 = candidate_thresholds[j]

            P1 = persistence_geometric_maps[t1]
            P2 = persistence_geometric_maps[t2]

            # ------------------------------------------
            # Binary masks
            # ------------------------------------------

            mask1 = P1.gte(cutoff)
            mask2 = P2.gte(cutoff)

            # ------------------------------------------
            # Intersection
            # ------------------------------------------

            intersection = (
                mask1
                .And(mask2)
                .rename("intersection")
            )

            # ------------------------------------------
            # Union
            # ------------------------------------------

            union = (
                mask1
                .Or(mask2)
                .rename("union")
            )

            # ------------------------------------------
            # Areas
            # ------------------------------------------

            intersection_area = (
                ee.Image.pixelArea()
                .updateMask(intersection)
                .reduceRegion(
                    reducer=ee.Reducer.sum(),
                    geometry=roi,
                    scale=500,
                    maxPixels=1e9
                )
                .get("area")
            )

            union_area = (
                ee.Image.pixelArea()
                .updateMask(union)
                .reduceRegion(
                    reducer=ee.Reducer.sum(),
                    geometry=roi,
                    scale=500,
                    maxPixels=1e9
                )
                .get("area")
            )

            area1 = (
                ee.Image.pixelArea()
                .updateMask(mask1)
                .reduceRegion(
                    reducer=ee.Reducer.sum(),
                    geometry=roi,
                    scale=500,
                    maxPixels=1e9
                )
                .get("area")
            )

            area2 = (
                ee.Image.pixelArea()
                .updateMask(mask2)
                .reduceRegion(
                    reducer=ee.Reducer.sum(),
                    geometry=roi,
                    scale=500,
                    maxPixels=1e9
                )
                .get("area")
            )

            # ------------------------------------------
            # Convert to km²
            # ------------------------------------------

            intersection_km2 = (
                ee.Number(intersection_area)
                .divide(1e6)
            )

            union_km2 = (
                ee.Number(union_area)
                .divide(1e6)
            )

            area1_km2 = (
                ee.Number(area1)
                .divide(1e6)
            )

            area2_km2 = (
                ee.Number(area2)
                .divide(1e6)
            )

            # ------------------------------------------
            # Jaccard
            # ------------------------------------------

            jaccard = (
                intersection_km2
                .divide(union_km2)
            )

            # ------------------------------------------
            # Retention
            # ------------------------------------------

            retention_t1 = (
                intersection_km2
                .divide(area1_km2)
                .multiply(100)
            )

            retention_t2 = (
                intersection_km2
                .divide(area2_km2)
                .multiply(100)
            )

            # ------------------------------------------
            # Get values
            # ------------------------------------------

            values = ee.Dictionary({
                "intersection_km2": intersection_km2,
                "union_km2": union_km2,
                "jaccard": jaccard,
                "area_t1_km2": area1_km2,
                "area_t2_km2": area2_km2,
                "retention_t1_percent": retention_t1,
                "retention_t2_percent": retention_t2
            }).getInfo()

            print(
                f"Threshold {t1} vs {t2} | "
                f"Intersection = {values['intersection_km2']:.2f} km² | "
                f"Union = {values['union_km2']:.2f} km² | "
                f"Jaccard = {values['jaccard']:.3f} | "
                f"Retention T{t1} = {values['retention_t1_percent']:.2f}% | "
                f"Retention T{t2} = {values['retention_t2_percent']:.2f}%"
            )

            persistence_robustness_results.append({
                "persistence_cutoff": cutoff,
                "threshold_1": t1,
                "threshold_2": t2,
                "intersection_km2": values["intersection_km2"],
                "union_km2": values["union_km2"],
                "jaccard": values["jaccard"],
                "retention_t1_percent": values["retention_t1_percent"],
                "retention_t2_percent": values["retention_t2_percent"]
            })


persistence_robustness_df = pd.DataFrame(
    persistence_robustness_results
)

print("\n")
print(
    persistence_robustness_df.to_string(
        index=False
    )
)

In [ ]:
# ==========================================
# Cell 44 — Final Temporal Persistence
# Main Scenario
# Radiance Threshold = 5
# Persistence = Geometric Mean
# ==========================================

final_radiance_threshold = 5

# ------------------------------------------
# Temporal components
# ------------------------------------------

final_frequency = (
    persistence_frequency_valid[
        final_radiance_threshold
    ]
    .rename("Persistence_Frequency")
)

final_intensity = (
    intensity_normalized[
        final_radiance_threshold
    ]
    .rename("Persistence_Intensity")
)

final_stability = (
    stability_maps[
        final_radiance_threshold
    ]
    .rename("Persistence_Stability")
)

# ------------------------------------------
# Final Geometric Persistence
# ------------------------------------------

final_persistence = (
    final_frequency
    .multiply(final_intensity)
    .multiply(final_stability)
    .pow(
        ee.Number(1).divide(3)
    )
    .rename("Temporal_Persistence")
)

print("Final temporal persistence created successfully.")
print(
    "Radiance threshold:",
    final_radiance_threshold
)
print(
    "Persistence method: Geometric Mean"
)

In [ ]:
# ==========================================
# Cell 45 — Final Persistence Statistics
# ==========================================

final_persistence_stats = final_persistence.reduceRegion(
    reducer=ee.Reducer.percentile(
        [1, 5, 10, 25, 50, 75, 90, 95, 99]
    ),
    geometry=roi,
    scale=500,
    maxPixels=1e9
).getInfo()

print("=" * 70)
print("FINAL TEMPORAL PERSISTENCE STATISTICS")
print("=" * 70)

for p in [1, 5, 10, 25, 50, 75, 90, 95, 99]:

    key = f"Temporal_Persistence_p{p}"

    print(
        f"P{p}: "
        f"{final_persistence_stats[key]:.4f}"
    )

print("\n" + "=" * 70)
print("AREA BY PERSISTENCE LEVEL")
print("=" * 70)

for cutoff in [0.3, 0.5, 0.7, 0.9]:

    mask = final_persistence.gte(cutoff)

    area = (
        ee.Image.pixelArea()
        .updateMask(mask)
        .reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=roi,
            scale=500,
            maxPixels=1e9
        )
        .get("area")
    )

    area_km2 = (
        ee.Number(area)
        .divide(1e6)
        .getInfo()
    )

    print(
        f"Persistence >= {cutoff:.1f}: "
        f"{area_km2:.2f} km²"
    )

In [ ]:
# ==========================================
# Cell 46 — Final Temporal Persistence Map
# With Legend
# ==========================================

Map_final_persistence = geemap.Map(
    basemap="SATELLITE"
)

Map_final_persistence.centerObject(
    roi,
    8
)

# ------------------------------------------
# Visualization parameters
# ------------------------------------------

persistence_vis = {
    "min": 0,
    "max": 1,
    "palette": [
        "000000",
        "313695",
        "74add1",
        "fee090",
        "f46d43",
        "a50026"
    ]
}

# ------------------------------------------
# Add Persistence layer
# ------------------------------------------

Map_final_persistence.addLayer(
    final_persistence,
    persistence_vis,
    "Temporal Persistence (2015–2025)"
)

# ------------------------------------------
# Add Tehran Province boundary
# ------------------------------------------

Map_final_persistence.addLayer(
    tehran.style(
        color="white",
        fillColor="00000000",
        width=2
    ),
    {},
    "Tehran Province Boundary"
)

# ------------------------------------------
# Legend
# ------------------------------------------

legend = """
<div style="
    background-color: white;
    padding: 10px;
    border: 2px solid grey;
    border-radius: 5px;
    font-size: 13px;
">
<b>Temporal Persistence</b><br>
<span style="
    display:inline-block;
    width:20px;
    height:12px;
    background:#000000;
"></span> 0.0 – 0.1<br>

<span style="
    display:inline-block;
    width:20px;
    height:12px;
    background:#313695;
"></span> 0.1 – 0.3<br>

<span style="
    display:inline-block;
    width:20px;
    height:12px;
    background:#74add1;
"></span> 0.3 – 0.5<br>

<span style="
    display:inline-block;
    width:20px;
    height:12px;
    background:#fee090;
"></span> 0.5 – 0.7<br>

<span style="
    display:inline-block;
    width:20px;
    height:12px;
    background:#f46d43;
"></span> 0.7 – 0.9<br>

<span style="
    display:inline-block;
    width:20px;
    height:12px;
    background:#a50026;
"></span> 0.9 – 1.0
</div>
"""

Map_final_persistence.add_html(
    legend,
    position="bottomright"
)

Map_final_persistence.add_layer_control()

Map_final_persistence